In [1]:
from google.colab import files
uploaded = files.upload()


Saving BIKE DETAILS.csv to BIKE DETAILS.csv


In [3]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("BIKE DETAILS.csv")
print(df.head())
print(f"\nAnzahl Einträge: {len(df)}")
print(f"\nSpalten: {df.columns.tolist()}")
print(f"\nFehlende Werte:\n{df.isnull().sum()}")
print(f"\nPreis-Statistik:\n{df['selling_price'].describe()}")

                                  name  selling_price  year seller_type  \
0            Royal Enfield Classic 350         175000  2019  Individual   
1                            Honda Dio          45000  2017  Individual   
2  Royal Enfield Classic Gunmetal Grey         150000  2018  Individual   
3    Yamaha Fazer FI V 2.0 [2016-2018]          65000  2015  Individual   
4                Yamaha SZ [2013-2014]          20000  2011  Individual   

       owner  km_driven  ex_showroom_price  
0  1st owner        350                NaN  
1  1st owner       5650                NaN  
2  1st owner      12000           148114.0  
3  1st owner      23000            89643.0  
4  2nd owner      21000                NaN  

Anzahl Einträge: 1061

Spalten: ['name', 'selling_price', 'year', 'seller_type', 'owner', 'km_driven', 'ex_showroom_price']

Fehlende Werte:
name                   0
selling_price          0
year                   0
seller_type            0
owner                  0
km_driven   

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

# 1. Marke extrahieren aus Name
df['brand'] = df['name'].apply(lambda x: x.split()[0])

# Royal Enfield und Harley-Davidson fix (zwei Wörter)
df.loc[df['name'].str.startswith('Royal Enfield'), 'brand'] = 'Royal Enfield'
df.loc[df['name'].str.startswith('Harley-Davidson'), 'brand'] = 'Harley-Davidson'

print("Marken:")
print(df['brand'].value_counts())

# 2. Owner als Zahl kodieren
owner_map = {
    '1st owner': 1,
    '2nd owner': 2,
    '3rd owner': 3,
    '4th owner': 4,
    '5th owner': 5
}
df['owner_num'] = df['owner'].map(owner_map).fillna(3)

# 3. Alter berechnen
df['age'] = 2024 - df['year']

# 4. Brand als Zahl kodieren
brand_codes = {brand: i for i, brand in enumerate(sorted(df['brand'].unique()))}
df['brand_code'] = df['brand'].map(brand_codes)
print(f"\nBrand Codes: {brand_codes}")

# 5. Features und Target definieren
features = ['brand_code', 'year', 'km_driven', 'owner_num', 'age']
X = df[features]
y = df['selling_price']

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining: {len(X_train)} | Test: {len(X_test)}")

# 7. Modell 1: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)
print(f"\n Random Forest:")
print(f"   MAE: {rf_mae:.0f}")
print(f"   R²:  {rf_r2:.4f}")

# 8. Modell 2: Gradient Boosting (Vergleich – Anforderung!)
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)
print(f"\n Gradient Boosting:")
print(f"   MAE: {gb_mae:.0f}")
print(f"   R²:  {gb_r2:.4f}")

# 9. Bestes Modell wählen und speichern
if gb_r2 > rf_r2:
    best_model = gb
    best_name = "Gradient Boosting"
else:
    best_model = rf
    best_name = "Random Forest"
print(f"\n✅ Bestes Modell: {best_name}")

# 10. Speichern
with open("motorcycle_price_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

# Brand codes auch speichern (brauchen wir in der App)
with open("brand_codes.pkl", "wb") as f:
    pickle.dump(brand_codes, f)

print("✅ Modell gespeichert!")
print(f"✅ Brand Codes gespeichert: {brand_codes}")

Marken:
brand
Bajaj              260
Hero               232
Honda              204
Royal Enfield      109
Yamaha             102
TVS                 69
Suzuki              30
KTM                 24
Mahindra             6
Vespa                4
Kawasaki             4
Jawa                 3
Activa               3
UM                   3
Aprilia              2
Harley-Davidson      2
BMW                  1
Benelli              1
Yo                   1
Hyosung              1
Name: count, dtype: int64

Brand Codes: {'Activa': 0, 'Aprilia': 1, 'BMW': 2, 'Bajaj': 3, 'Benelli': 4, 'Harley-Davidson': 5, 'Hero': 6, 'Honda': 7, 'Hyosung': 8, 'Jawa': 9, 'KTM': 10, 'Kawasaki': 11, 'Mahindra': 12, 'Royal Enfield': 13, 'Suzuki': 14, 'TVS': 15, 'UM': 16, 'Vespa': 17, 'Yamaha': 18, 'Yo': 19}

Training: 848 | Test: 213

 Random Forest:
   MAE: 19135
   R²:  0.5511

 Gradient Boosting:
   MAE: 17764
   R²:  0.6335

✅ Bestes Modell: Gradient Boosting
✅ Modell gespeichert!
✅ Brand Codes gespeichert: {'Activa

In [5]:
from google.colab import files
files.download("motorcycle_price_model.pkl")
files.download("brand_codes.pkl")
print("✅ Beide Dateien heruntergeladen!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Beide Dateien heruntergeladen!
